# PEAS – Vom Agentenmodell zur sinnvollen Entscheidung

## Szenario: Medikamenten-Lieferroboter im Krankenhaus

Ein autonomer Lieferroboter transportiert Medikamente innerhalb eines Krankenhauses.  
An einer Kreuzung muss er sich **lokal** für einen von drei Korridoren entscheiden.

```text
                 Korridor A
              d=4  p=4  e=1
                    ^
                    |
Korridor B  <---- ROBOT ---->  Korridor C
d=6 p=1 e=2                   d=5 p=2 e=4
```

Die Korridore unterscheiden sich hinsichtlich:

- **Distanz** zum Ziel,
- **Personenaufkommen**,
- **Energiebedarf**.

> **Wichtig:** Es geht hier **nicht um Wegsuche**. Der Agent wählt nur zwischen den aktuell angebotenen Alternativen A, B und C.

### Lernziele

Nach der Übung können Sie

1. ein Agentenszenario mit **PEAS** beschreiben,
2. erklären, warum das **Performance Measure** bestimmt, was für einen rationalen Agenten „gut“ ist,
3. erklären, wie die **Sensorik** die möglichen Entscheidungen eines Agenten begrenzt.




## 1. PEAS

Der Roboter steht an einer Kreuzung und kann einen der drei Korridore **A**, **B** oder **C** wählen.

| Korridor | Distanz | Personenaufkommen | Energiebedarf |
|---|---:|---:|---:|
| A | 4 | hoch | niedrig |
| B | 6 | niedrig | mittel |
| C | 5 | mittel | hoch |

### Aufgabe 1

Beschreiben Sie das Szenario mit dem **PEAS-Schema**.

- **P – Performance Measure:** Woran erkennt man, dass der Roboter seine Aufgabe gut erfüllt?
- **E – Environment:** Welche Eigenschaften der Umgebung sind relevant?
- **A – Actuators:** Welche Handlungen kann der Roboter ausführen?
- **S – Sensors:** Welche Informationen müsste der Roboter wahrnehmen können?


### Lösung zu Aufgabe 1

TODO

**P – Performance Measure:** 

**E – Environment:** 

**A – Actuators:** 

**S – Sensors:** 


In [2]:
import pandas as pd

# Drei verfügbare Korridore.
# Kleine Werte sind jeweils günstig.
BASE_CORRIDORS = [
    {"corridor": "A", "distance": 4, "occupancy": 4, "energy": 1},
    {"corridor": "B", "distance": 6, "occupancy": 1, "energy": 2},
    {"corridor": "C", "distance": 5, "occupancy": 2, "energy": 4},
]

corridors = BASE_CORRIDORS
pd.DataFrame(corridors)


,corridor,distance,occupancy,energy
0,A,4,4,1
1,B,6,1,2
2,C,5,2,4


## 2. Ein sehr einfacher Agent

Der erste Agent verwendet **nur die Distanz** und wählt den kürzesten Korridor.

Die Entscheidungsregel lautet:

> **Wähle die Alternative mit der kleinsten wahrgenommenen Distanz.**

### Aufgabe 2

1. Welchen Korridor wählt der Agent?
2. Warum ist diese Entscheidung nicht automatisch „gut“, obwohl sie bezüglich der Distanz sinnvoll ist?


In [3]:
def choose_shortest(corridors):
    """Wählt den Korridor mit der kleinsten Distanz."""
    return min(corridors, key=lambda c: c["distance"])["corridor"]

choice = choose_shortest(corridors)
print("Entscheidung des Distanz-Agenten:", choice)


Entscheidung des Distanz-Agenten: A


### Lösung zu Aufgabe 2

TODO


## 3. Das Performance Measure bestimmt, was „gut“ bedeutet

Wir verwenden eine einfache gewichtete Kostenfunktion:

$
J =
w_D \cdot \text{Distanz}
+
w_P \cdot \text{Personenaufkommen}
+
w_E \cdot \text{Energiebedarf}
$

**Je kleiner \(J\), desto besser.**

Die Gewichte legen fest, welche Ziele besonders wichtig sind.


In [3]:
def performance_cost(corridor, weights):
    """Berechnet die Kosten eines Korridors für gegebene Gewichtungen"""
    return (
        weights.get("distance", 0) * corridor["distance"] # das ", 0)" sorgt dafür, dass bei Nicht-Vorhandensein eine 0 returniert wird
        + weights.get("occupancy", 0) * corridor["occupancy"]
        + weights.get("energy", 0) * corridor["energy"]
    )


def compare_performance_measures(performance_measures):
    """Zeigt Kosten und bevorzugten Korridor für mehrere Performance Measures"""
    rows = []

    for name, weights in performance_measures.items():
        for corridor in corridors:
            rows.append({
                "Performance Measure": name,
                "Korridor": corridor["corridor"],
                "Kosten": performance_cost(corridor, weights),
            })

    table = pd.DataFrame(rows).pivot(
        index="Korridor",
        columns="Performance Measure",
        values="Kosten"
    )
    display(table)

    print("\nBevorzugter Korridor:")
    for name, weights in performance_measures.items():
        best = min(corridors, key=lambda c: performance_cost(c, weights))
        print(f"- {name}: Korridor {best['corridor']}")


### Aufgabe 3 – Performance Measures entwickeln

Definieren Sie **zwei unterschiedliche Performance Measures**:

1. **kurz & schnell:** Distanz steht klar im Vordergrund,
2. **sicher & ausgewogen:** Personenaufkommen soll stärker berücksichtigt werden; Distanz und Energie sollen aber nicht völlig ignoriert werden.

Tragen Sie Ihre Gewichte in die nächste Codezelle ein.

Diskutieren Sie anschließend kurz:

- Welcher Korridor wird jeweils bevorzugt?
- Warum ändert sich die Entscheidung?
- Welche Schwierigkeit entsteht grundsätzlich bei der Wahl solcher Gewichte?

#### Challenge

Finden Sie Gewichtungen, so dass **A**, **B** und **C** jeweils einmal die beste Alternative werden.


In [5]:
# TODO: Tragen Sie Ihre Gewichtungen ein.
# Beispielsyntax:
# "Name": {"distance": ..., "occupancy": ..., "energy": ...}

PERFORMANCE_MEASURES = {
    "kurz & schnell": {
        "distance": 1,
        "occupancy": 0,
        "energy": 0,
    },

    "sicher & ausgewogen": {
        # TODO
        "distance": ...,
        "occupancy": ...,
        "energy": ...,
    },
}

# Ersetzen Sie die ... durch Zahlen und führen Sie anschließend die Zelle aus
# compare_performance_measures(PERFORMANCE_MEASURES)


## 4. Gute Ziele reichen nicht – Sensorik begrenzt die Entscheidung

Nun verwenden wir für die **externe Bewertung** bei beiden Agenten dasselbe Performance Measure:

$
J =
1\cdot \text{Distanz}
+
3\cdot \text{Personenaufkommen}
+
2\cdot \text{Energiebedarf}
$

Wir vergleichen:

- **Agent 1:** nimmt nur die Distanz wahr und wählt den kürzesten Weg.
- **Agent 2:** nimmt alle drei Merkmale wahr und minimiert das vorgegebene Performance Measure.

Damit verändern wir nur einen PEAS-Bestandteil: **S – Sensors**.


In [9]:
BALANCED_PERFORMANCE = {
    "distance": 1,
    "occupancy": 3,
    "energy": 2
}

# Agent 1: sieht nur die Distanz.
choice_distance = min(corridors, key=lambda c: c["distance"])

# Agent 2: kennt alle Merkmale und kann das Performance Measure direkt anwenden.
choice_full = min(
    corridors,
    key=lambda c: performance_cost(c, BALANCED_PERFORMANCE)
)

comparison = pd.DataFrame([
    {
        "Agent": "nur Distanzsensor",
        "Wahl": choice_distance["corridor"],
        "tatsächliche Kosten": performance_cost(
            choice_distance, BALANCED_PERFORMANCE
        ),
    },
    {
        "Agent": "volle Sensorik",
        "Wahl": choice_full["corridor"],
        "tatsächliche Kosten": performance_cost(
            choice_full, BALANCED_PERFORMANCE
        ),
    },
])

comparison


,Agent,Wahl,tatsächliche Kosten
0,nur Distanzsensor,A,18
1,volle Sensorik,B,13


### Aufgabe 4 – Welche Rolle spielen die Sensoren?

1. Warum treffen die beiden Agenten unterschiedliche Entscheidungen, obwohl ihre Entscheidungen anschließend nach **demselben Performance Measure** bewertet werden?
2. Kann Agent 1 die fehlenden Informationen durch eine „intelligentere“ Entscheidungsregel vollständig kompensieren? Begründen Sie.


### Lösung zu Aufgabe 4

TODO


## 5. Abschluss – zwei Take-away-Fragen

Beantworten Sie jeweils in 2–3 Sätzen:

1. **Warum kann ein Agent trotz einer guten Entscheidungsregel schlechte Entscheidungen treffen?**
2. **Was ist der Unterschied zwischen Performance Measure und Entscheidungsregel?**


### Lösung / Take-away

TODO

---

### Aufgabe 5 (Optional)

Erweitern Sie das Szenario um **ein zusätzliches Merkmal**, z. B. `blocked` oder `noise`.

- Gehört das Merkmal zu **P, E, A oder S**?
- Wie müsste das Performance Measure bzw. die Wahrnehmung angepasst werden?
- Kann sich dadurch die bevorzugte Aktion ändern?


### Lösung zu Aufgabe 5

TODO

In [10]:
# TODO

*Quellhinweis:* Bei der Erstellung und Überarbeitung dieses Notebooks kam unterstützend generative KI zum Einsatz.
